# 01 - Bronze Ingestion

Loads the raw CSV files from `Files/raw/` into Bronze Delta tables **as-is**.

- Attach this notebook to `RetailLakehouse` (default Lakehouse).
- Every column is kept as a string so Bronze is a faithful copy of the source; typing happens in Silver.
- Two audit columns are added: `_ingested_at` and `_source_file`.

In [ ]:
from pyspark.sql import functions as F

RAW_PATH = "Files/raw"
sources = {
    "bronze_customers": f"{RAW_PATH}/customers.csv",
    "bronze_products": f"{RAW_PATH}/products.csv",
    "bronze_sales": f"{RAW_PATH}/sales.csv",
}

for table_name, file_path in sources.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)   # keep raw values as strings; Silver applies types
        .csv(file_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))  # replaces deprecated input_file_name()
    )
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    # Count from the Delta table (uses table statistics) instead of re-reading the CSV
    print(f"{table_name}: {spark.table(table_name).count():,} rows loaded")